In [2]:
import os
import json
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
import logging

# Logging Configuration
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Load OpenAI CLIP Model
repo = "openai/clip-vit-large-patch14"
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained(repo).to(device)
clip_processor = CLIPProcessor.from_pretrained(repo)

# Folder Paths
AUGMENT_KFASHION_IMAGE_FOLDER = "../Data/Augment/Augment_KFashion_Image"
AUGMENT_CLIP_FOLDER = "../Data/Augment/Augment_CLIP_v1"

In [ ]:
# Define attribute categories
categories = {
    "style": ["Street", "Modern", "Classic", "Feminine", "Casual", "Sporty", "Vintage", "Elegant", "Minimal"],
    "color": ["Black", "White", "Red", "Blue", "Green", "Yellow", "Pink", "Gray", "Beige"],
    "pattern": ["Solid", "Striped", "Checkered", "Floral", "Leopard", "Abstract"],
    "occasion": ["Casual Brunch", "Formal Event", "Park Picnic", "Office Meeting"],
    "season": ["Spring", "Summer", "Autumn", "Winter"],
    "clothing": ["Outerwear", "Top", "Bottoms", "Dress"],
    "length": ["Mini", "Midi", "Maxi"],
    "collar": ["Shirt Collar", "V-neck", "Round neck"],
    "sleeve": ["Sleeveless", "Short Sleeve", "3/4 Sleeve", "Long Sleeve"],
    "fit": ["Loose", "Fitted", "Oversized"]
}

def extract_attributes(image, category_options):
    inputs = clip_processor(text=category_options, images=image, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)
    return [category_options[i] for i in probs.argmax(dim=1)]

def process_image(image_path):
    image = Image.open(image_path).convert("RGB")
    attributes = {category: extract_attributes(image, options)[0] for category, options in categories.items()}
    return attributes

def generate_instruction_content(attributes):
    clothing_item = attributes['clothing']

    clothing_details = {
        "Length": attributes['length'],
        "Color": attributes['color'],
        "Category": clothing_item,
        "Collar": attributes['collar'],
        "Sleeve Length": attributes['sleeve'],
        "Print": attributes['pattern'],
        "Fit": attributes['fit']
    }

    input_caption = (
        f"Style: {attributes['style']}, "
        f"Outerwear: {clothing_details if clothing_item == 'Outerwear' else '{}'}, "
        f"Bottoms: {clothing_details if clothing_item == 'Bottoms' else '{}'}, "
        f"Dress: {clothing_details if clothing_item == 'Dress' else '{}'}, "
        f"Top: {clothing_details if clothing_item == 'Top' else '{}'}"
    )

    add_info = (
        f"Suitable occasion: {attributes['occasion']}, "
        f"Suitable season: {attributes['season']}, "
        f"Pattern: {attributes['pattern']}, "
        f"Shoes: Unknown, Accessories: None"
    )

    return input_caption, add_info

In [ ]:
for file_name in os.listdir(AUGMENT_KFASHION_IMAGE_FOLDER):
    if file_name.endswith(('.jpg', '.jpeg', '.png')):
        image_path = os.path.join(AUGMENT_KFASHION_IMAGE_FOLDER, file_name)
        try:
            # logging.info(f"Processing image: {file_name}")
            attributes = process_image(image_path)
            input_caption, add_info = generate_instruction_content(attributes)

            generated_instruction = {
                "Prompt": "Create a full-body photo of a 20-30s Korean woman in the specified outfit, excluding the face.",
                "Input": {"caption": input_caption},
                "Add_Info": add_info,
                "Output": file_name
            }

            save_file_name = f"{file_name.rsplit('.', 1)[0]}.json"
            save_path = os.path.join(AUGMENT_CLIP_FOLDER, save_file_name)

            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(generated_instruction, f, ensure_ascii=False, indent=4)
            logging.info(f"{save_file_name} saved in {AUGMENT_CLIP_FOLDER}.")
        except Exception as e:
            logging.error(f"Error processing {file_name}: {e}")

In [ ]:
len(os.listdir(AUGMENT_CLIP_FOLDER))